# 第 34 课 · SFT 数据、Assistant-only Loss 与多轮对话

**把第 33 课的“训练连接正确”，推进到“训练目标和数据处理正确”。**

本课产出：可检查的训练/验证划分、逐 token 监督标签、动态补齐批次、长度处理记录，以及分词器与数据契约。

| 运行层次 | 本课实际做什么 | 不代表什么 |
|---|---|---|
| 标准库离线管线 | 使用显式教学字节 codec，验证角色、标签、分组、长度和批次 | 不是 Qwen 分词结果 |
| 可选 PyTorch 实验 | 用简化的因果累积网络核对 loss 与梯度 | 不是 Transformer/预训练模型的能力评测 |
| 显式开启真实检查 | 固定版本 Qwen tokenizer、模板、offset 映射 | 不加载权重、不训练、不测 CUDA |

**制作时验证边界：**标准库和 CPU PyTorch 实验可执行；环境没有 Transformers，网络下载不可用。真实 tokenizer 单元默认关闭，没有模拟其成功结果。

Notebook 自包含，无需同目录存在 `.py`。脚本另行提供，可以独立运行。顺序运行不等于完成后面的独立练习。

## 0. 环境与学习方式

本课先不安装新库、不调用显卡。先预测每个例子的标签，再运行。

新增背景：SFT（监督微调）仍然训练下一个 token 的条件概率；我们通过 **labels 的选择**，决定哪些目标参与损失。参考：[TRL SFT 原理与标签](https://huggingface.co/docs/trl/v0.23.1/sft_trainer)。

下面加载标准库和公共辅助函数。它们不下载、不创建输出目录。

In [1]:
from __future__ import annotations

import argparse
import copy
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import importlib.util
import json
import math
from pathlib import Path
import random
import re
import sys
import tempfile
import traceback
from typing import Callable

MODEL_ID = "Qwen/Qwen3-0.6B"
REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
IGNORE = -100
SEED = 3401
FORMAT = "lesson34-sft-data-v1"
SOURCES = {
    "chat_templates": "https://huggingface.co/docs/transformers/v4.57.3/en/chat_templating",
    "assistant_loss": "https://huggingface.co/docs/trl/v0.23.1/sft_trainer",
    "template_updates": "https://huggingface.co/docs/trl/chat_templates",
    "causal_loss": "https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/loss/loss_utils.py",
    "qwen_template": f"https://huggingface.co/{MODEL_ID}/blob/{REVISION}/tokenizer_config.json",
    "cross_entropy": "https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html",
}


def require(condition, message: str) -> None:
    if not bool(condition):
        raise ValueError(message)


def digest(value) -> str:
    text = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def write_jsonl(path: Path, rows) -> None:
    with path.open("w", encoding="utf-8", newline="\n") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")


def unique_object(pairs):
    out = {}
    for key, value in pairs:
        require(key not in out, f"JSON重复键：{key}")
        out[key] = value
    return out


def read_jsonl(path) -> list[dict]:
    records = []
    with Path(path).open(encoding="utf-8") as f:
        for number, line in enumerate(f, 1):
            require(bool(line.strip()), f"第{number}行为空；每行必须是一条JSON记录")
            try:
                records.append(json.loads(line, object_pairs_hook=unique_object))
            except (ValueError, json.JSONDecodeError) as exc:
                raise ValueError(f"第{number}行：{exc}") from exc
    audit_records(records)
    return records


def environment() -> dict:
    versions = {}
    for package in ("torch", "transformers", "tokenizers", "trl"):
        try:
            versions[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            versions[package] = None
    return {"python": sys.version, "packages": versions}

def simple_record(user="U", assistant="A", identity="sample", system=None):
    messages = [] if system is None else [{"role":"system", "content":system}]
    messages += [{"role":"user","content":user},{"role":"assistant","content":assistant}]
    return {"id":identity,"group_id":identity,"messages":messages}

def boundary_fixture(messages):
    """教学模板替身：只模拟后文改变旧assistant格式的风险，不是Qwen实测。"""
    last_user = max(i for i,m in enumerate(messages) if m["role"]=="user")
    chunks = []
    for i,m in enumerate(messages):
        body = m["content"]
        if m["role"]=="assistant" and i>last_user:
            body = "<think>\n\n</think>\n\n" + body.lstrip("\n")
        chunks.append("<|im_start|>"+m["role"]+"\n"+body+"<|im_end|>\n")
    return "".join(chunks)

## 1. 一条对话记录，为什么需要 `id` 和 `group_id`？

`id` 区分记录；`group_id` 表示共同来源。例如同一订单派生出两种提问方式，它们应属于同一个 group。先按来源划分，再做编码或样本派生。

本课只接收：可选开头 system，随后 user/assistant 交替，最后一条必须是非空 assistant。拒绝工具、多模态、空答案和不完整对话；保留正文的缩进、空格与换行。

**数据质量不限于格式。** 下一步实际微调前仍要核查授权/隐私、答案正确性、标签一致性、任务覆盖和近重复；本课代码不能替代这些人工检查。

In [2]:
def validate_record(record: dict) -> None:
    require(isinstance(record, dict), "记录必须是字典")
    require(set(record) == {"id", "group_id", "messages"}, "字段必须恰好为id/group_id/messages")
    for field in ("id", "group_id"):
        require(isinstance(record[field], str) and bool(record[field].strip()), f"{field}必须非空")
    messages = record["messages"]
    require(isinstance(messages, list) and len(messages) >= 2, "至少需要一轮完整问答")
    for i, message in enumerate(messages):
        require(isinstance(message, dict) and set(message) == {"role", "content"}, f"消息{i}字段错误")
        require(message["role"] in ("system", "user", "assistant"), f"消息{i}角色不支持")
        require(isinstance(message["content"], str) and bool(message["content"].strip()), f"消息{i}内容不能为空")
        # 验证可编码，但不strip/归一化真实内容，保留代码缩进和换行。
        try:
            message["content"].encode("utf-8", errors="strict")
        except UnicodeError as exc:
            raise ValueError(f"消息{i}不是有效UTF-8文本") from exc
    start = int(messages[0]["role"] == "system")
    require(len(messages[start:]) >= 2 and len(messages[start:]) % 2 == 0, "对话必须结束于完整assistant回答")
    for i, message in enumerate(messages[start:]):
        expected = "user" if i % 2 == 0 else "assistant"
        require(message["role"] == expected, f"第{start+i}条应为{expected}")


def audit_records(records: list[dict]) -> None:
    require(isinstance(records, list) and bool(records), "数据集不能为空")
    ids, contents = set(), set()
    for record in records:
        validate_record(record)
        require(record["id"] not in ids, f"重复id：{record['id']}")
        signature = digest(record["messages"])
        require(signature not in contents, "发现逐字重复对话：先人工确认来源并显式去重")
        ids.add(record["id"])
        contents.add(signature)


def prompt_keys(record: dict) -> set[str]:
    """每个assistant答案之前的完整消息前缀。仅发现精确重复，不做语义去重。"""
    return {digest(record["messages"][:i]) for i, m in enumerate(record["messages"])
            if m["role"] == "assistant"}


def audit_split(train: list[dict], valid: list[dict]) -> dict:
    require(bool(train) and bool(valid), "训练/验证集均须非空")
    audit_records(train + valid)
    train_groups = {r["group_id"] for r in train}
    valid_groups = {r["group_id"] for r in valid}
    require(train_groups.isdisjoint(valid_groups), "同一来源group跨越训练/验证集")
    train_prompts = set().union(*(prompt_keys(r) for r in train))
    valid_prompts = set().union(*(prompt_keys(r) for r in valid))
    require(train_prompts.isdisjoint(valid_prompts), "精确相同的答案前缀跨越两侧；检查group_id")
    return {"train_records": len(train), "valid_records": len(valid),
            "train_groups": len(train_groups), "valid_groups": len(valid_groups),
            "group_overlap": 0, "exact_prompt_overlap": 0,
            "limitation": "不检测语义改写、未知来源关系或预训练数据污染"}


def split_by_group(records: list[dict], valid_fraction=0.25, seed=SEED):
    audit_records(records)
    require(type(valid_fraction) in (int, float) and 0 < valid_fraction < 1, "比例必须在0和1之间")
    groups = sorted({r["group_id"] for r in records})
    require(len(groups) >= 2, "至少需要两个独立来源group")
    random.Random(seed).shuffle(groups)  # 不改变调用者全局随机状态。
    n_valid = max(1, min(len(groups)-1, math.ceil(len(groups)*valid_fraction)))
    valid_groups = set(groups[:n_valid])
    train = sorted((copy.deepcopy(r) for r in records if r["group_id"] not in valid_groups), key=lambda r:r["id"])
    valid = sorted((copy.deepcopy(r) for r in records if r["group_id"] in valid_groups), key=lambda r:r["id"])
    audit_split(train, valid)
    return train, valid


def demo_records() -> list[dict]:
    """12个订单来源，每个来源有两份不同的两轮对话；共24条。不是质量基准。"""
    rows = []
    for g in range(12):
        item = ["红茶", "大米", "铅笔"][g % 3]
        quantity, price = g % 4 + 1, g % 5 + 2
        total = quantity * price
        first = f"订单R{g:02d}：{item}{quantity}份，单价{price}元。总价多少？"
        for variant in range(2):
            final_user = "只输出总价的整数。" if variant == 0 else "把总价写为JSON，键名为total。"
            final_answer = str(total) if variant == 0 else json.dumps({"total":total}, ensure_ascii=False)
            rows.append({"id":f"R{g:02d}-{variant}", "group_id":f"order-R{g:02d}", "messages":[
                {"role":"system", "content":"按订单计算，保留单位。"},
                {"role":"user", "content":first},
                {"role":"assistant", "content":f"总价为{total}元。"},
                {"role":"user", "content":final_user},
                {"role":"assistant", "content":final_answer},
            ]})
    return rows

In [3]:
records = demo_records()
print("记录数：", len(records), "来源数：", len({r['group_id'] for r in records}))
print(json.dumps(records[0], ensure_ascii=False, indent=2))
validate_record(records[0])

记录数： 24 来源数： 12
{
  "id": "R00-0",
  "group_id": "order-R00",
  "messages": [
    {
      "role": "system",
      "content": "按订单计算，保留单位。"
    },
    {
      "role": "user",
      "content": "订单R00：红茶1份，单价2元。总价多少？"
    },
    {
      "role": "assistant",
      "content": "总价为2元。"
    },
    {
      "role": "user",
      "content": "只输出总价的整数。"
    },
    {
      "role": "assistant",
      "content": "2"
    }
  ]
}


### 1.1 按来源分组切分，而不是直接洗牌 JSONL 行

本课 12 个订单来源，每个来源有两条不同的两轮对话。按 group 划分后，同一订单的变体不会跨到验证侧。

代码还检查逐字重复对话，以及跨 split 的精确答案前缀。**不检测语义级改写，也不能发现没有标注出来的共同来源。**

按 group 控制划分是标准的数据组织思路，例如 [GroupShuffleSplit 官方说明](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)。本课手写实现，不依赖 scikit-learn。

In [4]:
train_records, valid_records = split_by_group(records)
print(json.dumps(audit_split(train_records, valid_records), ensure_ascii=False, indent=2))
try:
    audit_split(records[::2], records[1::2])
except ValueError as error:
    print("预期拒绝逐行拆分：", error)

{
  "train_records": 18,
  "valid_records": 6,
  "train_groups": 9,
  "valid_groups": 3,
  "group_overlap": 0,
  "exact_prompt_overlap": 0,
  "limitation": "不检测语义改写、未知来源关系或预训练数据污染"
}
预期拒绝逐行拆分： 同一来源group跨越训练/验证集


## 2. 先用可手算的 codec，明确监督什么

教学 codec 的每条消息是：`角色ID + UTF-8字节 + EOT`。它没有训练词表，也不是 Qwen tokenizer。

我们刻意设置 `PAD_ID == EOT_ID == 259`，检验后面的程序是否按**位置和真实长度**识别补齐，而不是把所有同编号的结束符误删。

本课监督契约：选中的 assistant 正文和结束符参与损失；角色头、system、user 和补齐位置不直接参与。

In [5]:
def labels_from_mask(ids: list[int], mask: list[int]) -> list[int]:
    require(isinstance(ids, list) and len(ids) >= 2, "input_ids至少两个位置")
    require(all(type(x) is int and x >= 0 for x in ids), "token id必须为非负整数，排除bool")
    require(isinstance(mask, list) and len(mask) == len(ids), "mask长度必须等于输入长度")
    require(all(type(x) in (bool, int) and x in (0, 1) for x in mask), "mask只能是0/1")
    require(not mask[0], "首位置没有前驱logit，不能成为监督目标")
    require(any(mask[1:]), "移位后无有效监督token；拒绝全忽略样本")
    return [token if chosen else IGNORE for token, chosen in zip(ids, mask)]


def select_mask(length: int, spans: list[dict], policy="all") -> list[int]:
    require(policy in ("all", "last"), "policy应为all/last")
    assistant = [s for s in spans if s["role"] == "assistant"]
    require(bool(assistant), "缺少assistant区间")
    chosen = assistant if policy == "all" else assistant[-1:]
    result = [0] * length
    for span in chosen:
        begin, end = span["target_start"], span["target_end"]
        require(0 < begin < end <= length, "监督区间非法")
        for p in range(begin, end):
            result[p] = 1
    return result


def make_row(record, ids, spans, policy, codec_contract) -> dict:
    mask = select_mask(len(ids), spans, policy)
    labels = labels_from_mask(ids, mask)
    return {"id":record["id"], "group_id":record["group_id"], "input_ids":ids,
            "attention_mask":[1]*len(ids), "labels":labels,
            "assistant_mask":mask, "spans":spans, "policy":policy,
            "codec_fingerprint":digest(codec_contract), "source_fingerprint":digest(record["messages"])}


class ToyChatCodec:
    """显式role token + UTF-8字节 + EOT。故意令PAD==EOT以检验按长度补齐。"""
    roles = {"system":256, "user":257, "assistant":258}
    eot_id = 259
    pad_id = 259
    vocab_size = 260

    def contract(self) -> dict:
        return {"kind":"teaching-byte-codec-not-Qwen", "version":1, "roles":self.roles,
                "eot_id":self.eot_id, "pad_id":self.pad_id, "vocab_size":self.vocab_size,
                "targets":"selected assistant UTF-8 body + EOT; headers excluded"}

    def encode(self, record, policy="all") -> dict:
        validate_record(record)
        ids, spans = [], []
        for i, message in enumerate(record["messages"]):
            header = len(ids)
            ids.append(self.roles[message["role"]])
            begin = len(ids)
            ids.extend(message["content"].encode("utf-8"))
            ids.append(self.eot_id)
            spans.append({"message_index":i, "role":message["role"], "header_start":header,
                          "target_start":begin, "target_end":len(ids), "eot_index":len(ids)-1})
        return make_row(record, ids, spans, policy, self.contract())

    def display(self, ids):
        reverse = {v:k for k,v in self.roles.items()}
        pieces, buf = [], bytearray()
        def flush():
            if buf:
                pieces.append(bytes(buf).decode("utf-8", errors="backslashreplace")); buf.clear()
        for token in ids:
            if 0 <= token <= 255:
                buf.append(token)
            else:
                flush(); pieces.append("<EOT>" if token == self.eot_id else f"<{reverse.get(token, '?')}>")
        flush()
        return "".join(pieces)


def supervised_texts(row: dict, codec) -> list[str]:
    result, active = [], []
    for token, label in zip(row["input_ids"], row["labels"]):
        if label != IGNORE:
            active.append(token)
        elif active:
            result.append(codec.display(active)); active = []
    if active:
        result.append(codec.display(active))
    return result

In [6]:
codec = ToyChatCodec()
small_record = simple_record("A", "B")
small_row = codec.encode(small_record)
print("input_ids：", small_row['input_ids'])
print("labels：   ", small_row['labels'])
print("attention：", small_row['attention_mask'])
print("显示：", codec.display(small_row['input_ids']))
print("监督片段：", supervised_texts(small_row, codec))

input_ids： [257, 65, 259, 258, 66, 259]
labels：    [-100, -100, -100, -100, 66, 259]
attention： [1, 1, 1, 1, 1, 1]
显示： <user>A<EOT><assistant>B<EOT>
监督片段： ['B<EOT>']


### 2.1 `labels` 与输入同位：不要提前移位

上一个例子的正确数据是：

```text
input_ids: [USER, 'A', EOT, ASSISTANT, 'B', EOT]
labels:    [-100,-100,-100, -100,      'B', EOT]
```

传给本课预期的 HF CausalLM 时，它在内部使 **位置 t 的 logits 预测 labels[t+1]**。因此，assistant 角色头末位置的 logits 用来预测第一个回答 token。

手工核对损失时使用 `logits[:, :-1]` 对 `labels[:, 1:]`；**不要再将已经移位的标签传给内部会移位的模型。** 原始依据：[固定版本的 ForCausalLMLoss](https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/loss/loss_utils.py)。

训练输入包含答案用于 teacher forcing，但因果注意力仍然阻止看到未来目标；输入中有答案不意味着可以关闭因果遮挡。

In [7]:
for label_position, label in enumerate(small_row['labels']):
    if label != IGNORE:
        print(f"logits位置 {label_position-1} -> 目标位置 {label_position} -> token {label}")

logits位置 3 -> 目标位置 4 -> token 66
logits位置 4 -> 目标位置 5 -> token 259


### 2.2 多轮对话有两个不同的学习目标

- `policy="all"`：学习全部 assistant 回答。
- `policy="last"`：只学习末轮回答；早期 assistant 仍保留为上下文。

这是本课的显式数据策略，不是宣称某个训练库默认提供同名开关。

如果将一段对话拆成多个前缀，每个前缀又监督所有历史答案，早期答案会被重复加权。`last` 可以避免这种派生方式下重复直接监督历史，但并不意味着它总是更好。先明确目标，再选择策略。

In [8]:
all_row = codec.encode(records[0], policy="all")
last_row = codec.encode(records[0], policy="last")
print("输入相同：", all_row['input_ids'] == last_row['input_ids'])
print("all监督：", supervised_texts(all_row, codec))
print("last监督：", supervised_texts(last_row, codec))
print("目标数：", sum(x != IGNORE for x in all_row['labels']),
      sum(x != IGNORE for x in last_row['labels']))

输入相同： True
all监督： ['总价为2元。<EOT>', '2<EOT>']
last监督： ['2<EOT>']
目标数： 19 2


## 3. 三种 mask，不要混成一个

| 机制 | 控制什么 | 作用对象 |
|---|---|---|
| 因果 mask | 不允许读取未来 | attention 的 query-key 关系 |
| padding attention mask | 不读取补齐 key | 真实输入与补齐位置 |
| loss mask / `-100` | 哪些目标参与交叉熵 | labels |

**assistant-only 不意味着把 user 的 attention mask 设成 0。** 用户问题是回答需要的条件，仍须保留。`-100` 是标签忽略值，不是输入 token；不要把它送进 Embedding。

下面先定义批次与手工 loss 核对函数，后面逐步使用。

In [9]:
def validate_row(row):
    require(isinstance(row, dict), "样本必须是字典")
    for key in ("input_ids", "labels", "attention_mask"):
        require(key in row and isinstance(row[key], list), f"缺少列表字段{key}")
    n = len(row["input_ids"])
    require(n >= 2 and len(row["labels"]) == n and len(row["attention_mask"]) == n, "长度不一致")
    require(all(type(x) is int and x >= 0 for x in row["input_ids"]), "非法input_ids")
    require(row["attention_mask"] == [1]*n, "collator接收未padding样本")
    require(all(type(y) is int and (y == IGNORE or y == x) for x,y in zip(row["input_ids"],row["labels"])),
            "labels必须与input_ids同位，不能预先移位")
    require(row["labels"][0] == IGNORE and any(y != IGNORE for y in row["labels"][1:]), "有效目标为空或首位被监督")


def collate_rows(rows, pad_id: int, multiple=1):
    require(isinstance(rows, list) and bool(rows), "不能整理空批次")
    require(type(pad_id) is int and pad_id >= 0, "pad_id必须为非负整数")
    require(type(multiple) is int and multiple >= 1, "multiple必须为正整数")
    for row in rows:
        validate_row(row)
    width = max(len(r["input_ids"]) for r in rows)
    width = math.ceil(width/multiple)*multiple
    batch = {"input_ids":[], "attention_mask":[], "labels":[]}
    for row in rows:
        n = len(row["input_ids"])
        batch["input_ids"].append(row["input_ids"][:] + [pad_id]*(width-n))
        batch["attention_mask"].append([1]*n + [0]*(width-n))
        batch["labels"].append(row["labels"][:] + [IGNORE]*(width-n))
    # 只返回模型接受的三个字段，不把id/spans/来源等传给model(**batch)。
    return batch


def batch_statistics(batch):
    n_slots = sum(len(x) for x in batch["input_ids"])
    n_input = sum(sum(x) for x in batch["attention_mask"])
    n_targets = sum(sum(y != IGNORE for y in row[1:]) for row in batch["labels"])
    return {"batch_size":len(batch["input_ids"]), "width":len(batch["input_ids"][0]),
            "input_tokens":n_input, "supervised_targets":n_targets, "padding_slots":n_slots-n_input,
            "padding_fraction":(n_slots-n_input)/n_slots, "target_fraction":n_targets/n_input}


def merge_loss_summaries(summaries):
    require(bool(summaries), "loss摘要不能为空")
    total_loss, total_targets = 0.0, 0
    for loss_sum, count in summaries:
        require(type(count) is int and count > 0, "有效目标数必须为正整数")
        require(math.isfinite(float(loss_sum)) and loss_sum >= 0, "loss_sum必须有限非负")
        total_loss += loss_sum; total_targets += count
    return total_loss/total_targets


def causal_token_loss(logits, labels):
    """手工核对函数：内部执行一次移位。不要再把移位后的labels传给HF模型。"""
    import torch
    from torch.nn import functional as F
    require(logits.ndim == 3 and labels.ndim == 2 and logits.shape[:2] == labels.shape, "loss形状错误")
    require(labels.dtype == torch.long, "labels必须torch.long")
    targets = labels[:, 1:].contiguous()
    count = int((targets != IGNORE).sum().item())
    require(count > 0, "移位后的有效目标数为零")
    loss_sum = F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]), targets.reshape(-1),
                              ignore_index=IGNORE, reduction="sum")
    return loss_sum/count, loss_sum, count

### 3.1 不监督用户位置，用户表示还会收到梯度吗？

会有这种情况：答案预测使用了前面的用户表示，答案 loss 通过上下文计算传回去。

下面用“Embedding → 前缀均值 → Linear”的简化因果网络观察两件事：

1. 对忽略目标对应的 logits，直接梯度为零。
2. 用户位置的 embedding 表示仍然可能有非零梯度。

它不是 Transformer，也没有做质量训练。不要将其 loss 数值当作能力成绩。

In [10]:
def torch_demo():
    if importlib.util.find_spec("torch") is None:
        return {"status":"skipped", "reason":"未安装PyTorch"}
    import torch
    from torch import nn
    codec = ToyChatCodec()
    rec = {"id":"gradient", "group_id":"gradient", "messages":[
        {"role":"user", "content":"AB"}, {"role":"assistant", "content":"C"}]}
    row = codec.encode(rec)
    batch = {k:torch.tensor(v, dtype=torch.long) for k,v in collate_rows([row], codec.pad_id).items()}
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(SEED)
        embedding, head = nn.Embedding(codec.vocab_size, 8), nn.Linear(8, codec.vocab_size)
        embedded = embedding(batch["input_ids"])
        embedded.retain_grad()
        # 简化因果上下文聚合，仅为梯度实验；不是Transformer或预训练模型。
        hidden = embedded.cumsum(dim=1) / torch.arange(1,embedded.shape[1]+1).view(1,-1,1)
        logits = head(hidden); logits.retain_grad()
        loss, _, count = causal_token_loss(logits, batch["labels"])
        loss.backward()
        next_targets = batch["labels"][:, 1:]
        ignored_gradient = logits.grad[:, :-1][next_targets == IGNORE].abs().max().item()
        prompt_gradient = embedded.grad[0, 1:3].norm().item()
        original_loss = loss.item()
        altered = logits.detach().clone()
        altered[:, :-1][next_targets == IGNORE] += torch.linspace(-200,200,codec.vocab_size)
        same_loss = causal_token_loss(altered,batch["labels"])[0].item()
        return {"status":"passed", "scope":"CPU教学因果累积网络；不是Qwen训练或能力评测",
                "input_ids":row["input_ids"], "labels":row["labels"], "supervised_targets":count,
                "loss":original_loss, "ignored_logit_gradient_max":ignored_gradient,
                "prompt_embedding_gradient_norm":prompt_gradient,
                "loss_after_changing_ignored_logits":same_loss}

In [11]:
gradient_report = torch_demo()
print(json.dumps(gradient_report, ensure_ascii=False, indent=2))

{
  "status": "passed",
  "scope": "CPU教学因果累积网络；不是Qwen训练或能力评测",
  "input_ids": [
    257,
    65,
    66,
    259,
    258,
    67,
    259
  ],
  "labels": [
    -100,
    -100,
    -100,
    -100,
    -100,
    67,
    259
  ],
  "supervised_targets": 2,
  "loss": 5.523005485534668,
  "ignored_logit_gradient_max": 0.0,
  "prompt_embedding_gradient_norm": 0.07042372971773148,
  "loss_after_changing_ignored_logits": 5.523005485534668
}


## 4. 真实模板：不能假设“逐轮编码后拼起来”一定正确

某些模板会根据“最后一次 user 出现在哪里”，改变历史 assistant 的呈现。固定 Qwen3 模板中存在这样的分支。因此，较短对话的完整序列不一定是较长对话序列的严格前缀。

下面用**明确的教学替身**展示这一风险：只模拟末轮空 thinking 片段的变化，并不是实际 Qwen tokenizer 执行结果。真实依据可在 [固定 tokenizer_config](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json) 中查 `last_query_index`。

本课不会为了让拼接成功，偷偷重写用户数据或假装旧前缀稳定。

In [12]:
short_render = boundary_fixture(records[0]['messages'][:3])
long_render = boundary_fixture(records[0]['messages'])
print("短对话是否为完整对话前缀：", long_render.startswith(short_render))
print("短对话末尾：", repr(short_render[-85:]))
print("长对话历史回答附近：", repr(long_render[long_render.index('<|im_start|>assistant'):][:85]))

短对话是否为完整对话前缀： False
短对话末尾： '：红茶1份，单价2元。总价多少？<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n总价为2元。<|im_end|>\n'
长对话历史回答附近： '<|im_start|>assistant\n总价为2元。<|im_end|>\n<|im_start|>user\n只输出总价的整数。<|im_end|>\n<|im_star'


### 4.1 `return_assistant_tokens_mask=True` 不是自动识别所有模板的魔法

原生 assistant mask 需要模板的 `{% generation %}` / `{% endgeneration %}` 标记支持；不能只看返回对象存在就假定有效，还要检查长度、非零目标和 EOS 边界。

当前 TRL 文档描述了为部分模型自动替换训练模板的行为，**版本会影响实际选择**。本课不依赖这一自动行为，不引入 TRL，也不静默修补模板。参考：[模板能力要求](https://huggingface.co/docs/trl/v0.23.1/sft_trainer)、[当前训练模板说明](https://huggingface.co/docs/trl/chat_templates)。

本课真实路径限定为固定 Qwen3 的纯文本对话：一次渲染整段，严格核对角色/消息数/控制分隔符，然后用 fast tokenizer 的 offsets 映射正文和结束符。发现 token 跨监督边界、额外模板变换、工具或显式思考内容，就拒绝，而不是猜。

**真实路径监督序列化 assistant 正文，包括模板确实插入的空 thinking 片段，以及 `<|im_end|>`。角色头和结束后的换行不监督。** 这是一份明确契约；改用只监督最终答案或别的模板时，需要重新审查。

In [13]:
def qwen_char_spans(text: str, messages: list[dict]) -> list[dict]:
    """只用于本课固定Qwen纯文本模板。遇到未知格式明确失败，不猜边界。"""
    start_tag, end_tag = "<|im_start|>", "<|im_end|>"
    pos, spans = 0, []
    empty_think = "<think>\n\n</think>\n\n"
    for i, message in enumerate(messages):
        header = start_tag + message["role"] + "\n"
        require(text.startswith(header, pos), f"第{i}条消息模板头部不匹配")
        begin = pos + len(header)
        end = text.find(end_tag, begin)
        require(end >= 0, "模板缺少消息结束符")
        body = text[begin:end]
        require(start_tag not in body, "消息体包含控制分隔符")
        allowed = {message["content"]}
        if message["role"] == "assistant":
            # 固定Qwen模板可能给最后assistant自动加入空thinking区。
            allowed.add(empty_think + message["content"].lstrip("\n"))
        require(body in allowed, "序列化正文发生未支持的变换：不猜测、不中途替换模板")
        target_end = end + len(end_tag)
        spans.append({"message_index":i, "role":message["role"], "char_start":begin,
                      "char_end":target_end, "body":body})
        require(text.startswith("\n", target_end), "结束符后缺少预期换行")
        pos = target_end + 1
    require(pos == len(text), "模板有多余尾部；训练序列不可加下一assistant前缀")
    return spans


def offsets_to_token_spans(offsets, char_spans, ids, eot_id):
    """边界跨token时拒绝，避免悄悄把header/user监督进来。"""
    require(len(offsets) == len(ids), "offset长度错误")
    token_spans = []
    for span in char_spans:
        start, end = span["char_start"], span["char_end"]
        included = []
        for j, (a,b) in enumerate(offsets):
            if b <= a:
                continue
            if a < end and b > start:
                require(a >= start and b <= end, "token跨越监督边界；本课拒绝这种分词布局")
                included.append(j)
        require(bool(included), "模板区间没有映射到token")
        require(included == list(range(included[0], included[-1]+1)), "监督区间不连续")
        require(ids[included[-1]] == eot_id, "结束token不在监督区间内")
        token_spans.append({"message_index":span["message_index"], "role":span["role"],
                            "target_start":included[0], "target_end":included[-1]+1,
                            "eot_index":included[-1]})
    return token_spans


class QwenChatCodec:
    """真实tokenizer专用适配器。只加载tokenizer；不加载Qwen权重，也不依赖PEFT。"""
    def __init__(self, local_files_only=False):
        try:
            from transformers import AutoTokenizer
        except ImportError as exc:
            raise RuntimeError("真实分词器路径需要transformers；离线部分不需要。") from exc
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, use_fast=True,
                                                       trust_remote_code=False, local_files_only=local_files_only)
        require(self.tokenizer.is_fast, "本课需要fast tokenizer的offset_mapping")
        require(isinstance(self.tokenizer.chat_template, str), "模板必须为明确字符串")
        self.eot_id = self.tokenizer.convert_tokens_to_ids("<|im_end|>")
        require(type(self.eot_id) is int and self.tokenizer.encode("<|im_end|>", add_special_tokens=False)==[self.eot_id],
                "结束符必须是单独的已知特殊token")
        self.pad_id = self.tokenizer.pad_token_id
        require(type(self.pad_id) is int, "缺少pad_id；本课不自动改写tokenizer")
        self._contract = {"kind":"qwen-fixed-revision-real-tokenizer", "model_id":MODEL_ID, "revision":REVISION,
                          "template_sha256":digest(self.tokenizer.chat_template),
                          "vocab_sha256":digest(self.tokenizer.get_vocab()), "pad_id":self.pad_id, "eot_id":self.eot_id,
                          "enable_thinking":False, "add_generation_prompt":False,
                          "targets":"selected serialized assistant body (including template empty think if present) + im_end; header/trailing newline excluded",
                          "transformers":importlib.metadata.version("transformers")}

    def contract(self):
        return copy.deepcopy(self._contract)

    def encode(self, record, policy="all"):
        validate_record(record)
        tok = self.tokenizer
        forbidden = set(tok.all_special_tokens) | {"<think>", "</think>"}
        for m in record["messages"]:
            require(not any(s and s in m["content"] for s in forbidden),
                    "本课真实路径不支持正文中的保留控制串/显式thinking；不会靠扫描正文猜角色")
        kwargs = dict(add_generation_prompt=False, enable_thinking=False)
        text = tok.apply_chat_template(record["messages"], tokenize=False, **kwargs)
        full = tok.apply_chat_template(record["messages"], tokenize=True, **kwargs)
        enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)
        require(enc["input_ids"] == full, "直接模板与文本再编码的ID不一致")
        chars = qwen_char_spans(text, record["messages"])
        spans = offsets_to_token_spans(enc["offset_mapping"], chars, full, self.eot_id)
        row = make_row(record, full, spans, policy, self.contract())
        row["rendered_text"] = text
        row["character_spans"] = chars
        return row

    def display(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)

    def native_mask_audit(self, record):
        """只探测，不把未验证的native mask作为训练标签。"""
        template = self.tokenizer.chat_template
        if not re.search(r"\{%[-+]?\s*generation\b", template):
            return {"status":"unsupported_template", "reason":"未发现generation标记；不能把全零mask当成功"}
        try:
            value = self.tokenizer.apply_chat_template(record["messages"], tokenize=True, return_dict=True,
                        return_assistant_tokens_mask=True, add_generation_prompt=False, enable_thinking=False)
            expected = self.encode(record, "all")
            mask = value.get("assistant_masks")
            require(isinstance(mask, list) and len(mask)==len(expected["input_ids"]), "native mask缺失或长度错误")
            require(value["input_ids"] == expected["input_ids"] and any(mask), "native ID错误或mask全零")
            same = mask == expected["assistant_mask"]
            return {"status":"same_contract" if same else "different_contract_do_not_substitute",
                    "native_count":sum(mask), "our_count":sum(expected["assistant_mask"])}
        except Exception as exc:
            return {"status":"probe_failed_not_used", "error":f"{type(exc).__name__}: {exc}"}

## 5. 截断：不能只把 token 列表切到 `max_length`

直接 `ids[:max_length]` 可能只剩 prompt、切断答案或丢掉结束符。训练仍能运行，不代表样本还表达原来的任务。

本课提供：

| 策略 | 行为 |
|---|---|
| `drop`，默认 | 超长完整样本丢弃，并留下原因 |
| `keep_latest_rounds`，显式开启 | 只删除最老的完整 user+assistant 轮次，保留 system 和末轮，再重新渲染与编码 |

末轮仍放不下就丢弃。**保留整轮不保证保留语义**：末轮若说“根据刚才的订单”，删除历史可能破坏题意。因此默认不用裁历史，开启后必须检查记录。

长度处理之后，还会重新检查处理后两侧是否产生精确重复前缀。

In [14]:
def prepare_record(record, codec, max_length=256, policy="all", overflow="drop"):
    validate_record(record)
    require(type(max_length) is int and max_length >= 4, "max_length必须为至少4的整数")
    require(overflow in ("drop", "keep_latest_rounds"), "未知超长策略")
    working = copy.deepcopy(record)
    row = codec.encode(working, policy)
    original_length = len(row["input_ids"])
    removed = 0
    while len(row["input_ids"]) > max_length:
        messages = working["messages"]
        first = int(messages[0]["role"] == "system")
        if overflow == "drop" or len(messages)-first <= 2:
            return None, {"id":record["id"], "status":"dropped", "reason":"overlength_complete_sample",
                          "original_tokens":original_length, "remaining_tokens":len(row["input_ids"]),
                          "removed_rounds":removed}
        # 只删除最老一整轮，保留开头system和最后完整一轮，然后重新套模板编码。
        del messages[first:first+2]
        removed += 1
        row = codec.encode(working, policy)
    row["original_source_fingerprint"] = digest(record["messages"])
    row["removed_rounds"] = removed
    row["effective_messages"] = working["messages"]
    return row, {"id":record["id"], "status":"kept", "original_tokens":original_length,
                 "final_tokens":len(row["input_ids"]), "removed_rounds":removed,
                 "warning":"删除历史可能改变语义，必须人工检查" if removed else None}

In [15]:
long_record = {
    'id':'long', 'group_id':'long', 'messages':[
        {'role':'system','content':'S'},
        {'role':'user','content':'X'*20},
        {'role':'assistant','content':'old answer'},
        {'role':'user','content':'U'},
        {'role':'assistant','content':'A'},
    ]}
for overflow in ['drop', 'keep_latest_rounds']:
    row, event = prepare_record(long_record, codec, max_length=9, overflow=overflow)
    print(overflow, event)
    if row:
        print('保留文本：', codec.display(row['input_ids']))
        print('监督文本：', supervised_texts(row, codec))

drop {'id': 'long', 'status': 'dropped', 'reason': 'overlength_complete_sample', 'original_tokens': 43, 'remaining_tokens': 43, 'removed_rounds': 0}
keep_latest_rounds {'id': 'long', 'status': 'kept', 'original_tokens': 43, 'final_tokens': 9, 'removed_rounds': 1, 'warning': '删除历史可能改变语义，必须人工检查'}
保留文本： <system>S<EOT><user>U<EOT><assistant>A<EOT>
监督文本： ['A<EOT>']


## 6. 动态右补齐：按照原长度识别 padding

本课训练使用右补齐，补到当前批次最长长度。可选 `multiple` 只把长度向上对齐，不代表一定能加速。

当 PAD 与 EOT 编号相同，不能使用 `labels[input_ids == pad_id] = -100`，否则真实回答结束标记也会被删掉。类似地，不要使用 `input_ids != pad_id` 推断有效 attention。

**已有 labels 必须保留。** 后续若接入自动重建 labels 的普通语言模型 collator，可能把 assistant-only 掩码覆盖掉。只传 `input_ids`、`attention_mask`、`labels` 给模型，不把来源或 spans 传进去。

In [16]:
rows = [codec.encode(simple_record('A','B')), codec.encode(simple_record('longer','答案'))]
batch = collate_rows(rows, codec.pad_id)
for key,value in batch.items():
    print(key, value)
print('短样本真实EOT：', batch['labels'][0][len(rows[0]['input_ids'])-1])
print('短样本补齐尾部：', batch['labels'][0][len(rows[0]['input_ids']):])
print(batch_statistics(batch))

input_ids [[257, 65, 259, 258, 66, 259, 259, 259, 259, 259, 259, 259, 259, 259, 259, 259], [257, 108, 111, 110, 103, 101, 114, 259, 258, 231, 173, 148, 230, 161, 136, 259]]
attention_mask [[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]
labels [[-100, -100, -100, -100, 66, 259, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100], [-100, -100, -100, -100, -100, -100, -100, -100, -100, 231, 173, 148, 230, 161, 136, 259]]
短样本真实EOT： 259
短样本补齐尾部： [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
{'batch_size': 2, 'width': 16, 'input_tokens': 22, 'supervised_targets': 9, 'padding_slots': 10, 'padding_fraction': 0.3125, 'target_fraction': 0.4090909090909091}


### 6.1 有效 loss 分母是监督目标数，不是输入长度

本课按所有有效 assistant token 等权：

\[
L=\frac{\sum_{i,t} m_{i,t}\big[-\log p_\theta(x_{i,t}\mid x_{i,<t})\big]}{\sum_{i,t}m_{i,t}}
\]

长回答因此贡献更多目标。按每条样本等权也是一种设计，但那是不同目标，要明确声明。

两个批次分别有 9 和 1 个监督目标，均值为 1 和 9。直接平均批次均值是 5；按 token 加权应为 1.8。分母还要按实际内部移位后的目标检查，不能接受零监督样本。

In [17]:
print('错误：平均各batch均值 =', (1+9)/2)
print('正确：累计loss_sum/累计目标数 =', merge_loss_summaries([(9.0,9),(9.0,1)]))

错误：平均各batch均值 = 5.0
正确：累计loss_sum/累计目标数 = 1.8


## 7. Packing 为什么暂时关闭？

把样本拼进一条长序列可以减少 padding，但“插入 EOS”本身不会使普通因果注意力自动忘记前一个样本。第二个样本的 token 仍可能读取第一个样本。

下面比较普通下三角矩阵和增加 segment 隔离的矩阵。本课不实现高效 packing，不以此断言所有训练库的 packing 都会泄漏；后端可能提供专门的段边界处理。

In [18]:
def packing_demo():
    segments = [0,0,0,1,1]
    causal = [[j <= i for j in range(5)] for i in range(5)]
    block = [[j <= i and segments[j] == segments[i] for j in range(5)] for i in range(5)]
    return {"segment_ids":segments, "ordinary_causal":causal, "isolated_causal":block,
            "second_document_reads_first_without_boundary_mask":causal[3][0],
            "with_isolation":block[3][0],
            "note":"EOS只是token，普通因果mask不会自动跨文档重置；本课不启用packing"}


def prepare_dataset(records, codec, max_length=256, policy="all", overflow="drop", seed=SEED):
    train, valid = split_by_group(records, seed=seed)
    raw_splits, prepared, events = {"train":train,"valid":valid}, {}, []
    for split, subset in raw_splits.items():
        prepared[split] = []
        for record in subset:
            row, event = prepare_record(record,codec,max_length,policy,overflow)
            events.append({"split":split, **event})
            if row is not None:
                prepared[split].append(row)
        require(bool(prepared[split]), f"长度处理后{split}为空，请调整长度/数据")
    # 裁剪历史后再次查精确重复前缀：来源切分本身仍不足以防处理后的重合。
    effective = {split:[{"id":r["id"], "group_id":r["group_id"], "messages":r["effective_messages"]}
                       for r in rows] for split,rows in prepared.items()}
    post_audit = audit_split(effective["train"], effective["valid"])
    summary = {}
    for split, rows in prepared.items():
        summary[split] = {"records":len(rows), "input_tokens":sum(len(r["input_ids"]) for r in rows),
                          "supervised_targets":sum(sum(t != IGNORE for t in r["labels"][1:]) for r in rows),
                          "max_length":max(len(r["input_ids"]) for r in rows),
                          "removed_rounds":sum(r["removed_rounds"] for r in rows)}
    return {"raw_splits":raw_splits, "rows":prepared, "events":events, "summary":summary,
            "split_audit":post_audit, "codec_contract":codec.contract(),
            "data_signature":digest(records), "policy":policy, "max_length":max_length, "overflow":overflow}


def run_pipeline(output_dir="lesson_34_outputs", records=None, real=False, local_files_only=False,
                 max_length=256, policy="all", overflow="drop", include_torch=True):
    root = Path(output_dir)
    root.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    out = root / (("real_tokenizer_" if real else "offline_") + stamp)
    out.mkdir()
    manifest = {"format":FORMAT, "environment":environment(), "scope":"tokenizer_only_no_weights" if real else "teaching_codec",
                "status":"started", "sources":SOURCES}
    write_json(out/"manifest.json", manifest)
    try:
        records = demo_records() if records is None else records
        codec = QwenChatCodec(local_files_only) if real else ToyChatCodec()
        data = prepare_dataset(records,codec,max_length,policy,overflow)
        for split in ("train","valid"):
            write_jsonl(out/f"{split}_messages.jsonl",data["raw_splits"][split])
            write_jsonl(out/f"{split}_tokens.jsonl",data["rows"][split])
        write_json(out/"split_audit.json",data["split_audit"])
        write_json(out/"length_events.json",data["events"])
        contract = {"format":FORMAT, "codec":codec.contract(), "policy":policy, "max_length":max_length,
                    "overflow":overflow, "label_shift":"unshifted; CausalLM shifts internally", "packing":False,
                    "data_signature":data["data_signature"], "train_ids":[r["id"] for r in data["raw_splits"]["train"]],
                    "valid_ids":[r["id"] for r in data["raw_splits"]["valid"]]}
        contract["fingerprint"] = digest(contract)
        write_json(out/"contract.json",contract)
        inspected = data["rows"]["train"][:2]
        batch = collate_rows(inspected,codec.pad_id,multiple=8)
        report = {"summary":data["summary"], "batch":batch_statistics(batch), "packing":packing_demo(),
                  "inspected_rows":[{"id":r["id"], "supervised_texts":supervised_texts(r,codec),
                                     "labels":r["labels"], "input_ids":r["input_ids"]} for r in inspected],
                  "target_weighting_example":{"wrong_batch_mean":5.0,"correct_token_mean":merge_loss_summaries([(9.0,9),(9.0,1)])},
                  "gradient_demo":torch_demo() if include_torch else {"status":"disabled"}}
        if real:
            report["native_mask_probe"] = codec.native_mask_audit(records[0])
            codec.tokenizer.save_pretrained(out/"tokenizer")
        write_json(out/"batch_example.json",batch)
        write_json(out/"report.json",report)
        manifest.update(status="completed", contract_fingerprint=contract["fingerprint"], summary=data["summary"])
        write_json(out/"manifest.json",manifest)
        return out, report
    except Exception as exc:
        manifest.update(status="failed",error=f"{type(exc).__name__}: {exc}", traceback=traceback.format_exc())
        write_json(out/"manifest.json",manifest)
        raise

In [19]:
packing = packing_demo()
print('普通causal，第2个样本首位置可以读哪些key：', packing['ordinary_causal'][3])
print('加入隔离后：', packing['isolated_causal'][3])
print(packing['note'])

普通causal，第2个样本首位置可以读哪些key： [True, True, True, True, False]
加入隔离后： [False, False, False, True, False]
EOS只是token，普通因果mask不会自动跨文档重置；本课不启用packing


## 8. 同一份输入，监督策略不同，统计也不同

本课的模板数据由代码构造；训练/验证只用于检验数据管线，没有训练模型，也不报告“泛化成绩”。

下面对 `all` 与 `last` 使用完全相同的来源划分。比较输入 token 数与监督目标数，而不是只报告样本条数。

In [20]:
summaries = {}
for policy in ('all','last'):
    prepared = prepare_dataset(records, codec, policy=policy)
    summaries[policy] = prepared['summary']
print(json.dumps(summaries, ensure_ascii=False, indent=2))

{
  "all": {
    "train": {
      "records": 18,
      "input_tokens": 2739,
      "supervised_targets": 453,
      "max_length": 165,
      "removed_rounds": 0
    },
    "valid": {
      "records": 6,
      "input_tokens": 917,
      "supervised_targets": 155,
      "max_length": 165,
      "removed_rounds": 0
    }
  },
  "last": {
    "train": {
      "records": 18,
      "input_tokens": 2739,
      "supervised_targets": 141,
      "max_length": 165,
      "removed_rounds": 0
    },
    "valid": {
      "records": 6,
      "input_tokens": 917,
      "supervised_targets": 49,
      "max_length": 165,
      "removed_rounds": 0
    }
  }
}


### 8.1 完整导出

运行下一单元才会写文件。每次建立新的时间戳子目录，不覆盖旧结果。

| 产物 | 作用 |
|---|---|
| `train_messages.jsonl` / `valid_messages.jsonl` | 划分后的原始对话，便于审阅 |
| `train_tokens.jsonl` / `valid_tokens.jsonl` | 未移位标签、监督区间和实际采用的消息 |
| `contract.json` | codec/模板/词表身份、监督策略、长度规则、数据指纹 |
| `length_events.json` | 每个样本的保留、丢弃和删轮原因 |
| `batch_example.json` | 一份可检查的补齐结果 |
| `report.json` / `split_audit.json` | 统计、梯度实验和划分检查 |

教学 codec 导出会明确标注不是 Qwen。**不能把这些教学编号交给真实模型。** 文件含输入文本，分享前检查隐私。

In [21]:
OUTPUT_ROOT = 'lesson_34_outputs'
offline_dir, offline_report = run_pipeline(OUTPUT_ROOT)
print('输出目录：', offline_dir.resolve())
print(json.dumps(offline_report['summary'], ensure_ascii=False, indent=2))

输出目录： /mnt/data/_lesson34_work/notebook_run/lesson_34_outputs/offline_20261101T193933_021183Z
{
  "train": {
    "records": 18,
    "input_tokens": 2739,
    "supervised_targets": 453,
    "max_length": 165,
    "removed_rounds": 0
  },
  "valid": {
    "records": 6,
    "input_tokens": 917,
    "supervised_targets": 155,
    "max_length": 165,
    "removed_rounds": 0
  }
}


## 9. 可选：只加载真实 Qwen tokenizer

继续沿用第 33 课的环境。缺少 Transformers 时，可在独立环境使用：

```bash
python -m pip install "transformers==4.57.3"
```

这是代码参考的固定接口版本，不是本制作环境已实测的真实 tokenizer 兼容性结论。不要为本课重装 GPU 驱动或模型权重。

本课不需要 PEFT/TRL，不下载 Qwen 权重。首次检查仍需要下载 tokenizer 文件；已有缓存可以 `local_files_only=True`。

成功后保存实际 tokenizer、模板/词表指纹、监督片段和 native mask 探测。未支持的 native mask 不会被偷偷替换为全 1。

训练完整对话使用 `add_generation_prompt=False`；模板文本再次分词时使用 `add_special_tokens=False`。参考：[Transformers Chat templates](https://huggingface.co/docs/transformers/v4.57.3/en/chat_templating)。

In [22]:
RUN_REAL_INSPECTION = False
LOCAL_FILES_ONLY = False

if RUN_REAL_INSPECTION:
    real_dir, real_report = run_pipeline(
        OUTPUT_ROOT, real=True, local_files_only=LOCAL_FILES_ONLY,
        max_length=256, policy='all', overflow='drop',
    )
    print('真实tokenizer输出：', real_dir.resolve())
    print(json.dumps(real_report['summary'], ensure_ascii=False, indent=2))
    print(real_report['native_mask_probe'])
    for row in real_report['inspected_rows']:
        print(row['id'], row['supervised_texts'])
else:
    print('真实Qwen tokenizer路径未开启：不加载权重，不伪造真实检查结果。')

真实Qwen tokenizer路径未开启：不加载权重，不伪造真实检查结果。


### 9.1 换成自己的 JSONL 数据

每行恰好有 `id`、`group_id`、`messages`，例如：

```json
{"id":"order-01-v1","group_id":"order-01","messages":[{"role":"user","content":"2杯茶，每杯3元，总价多少？"},{"role":"assistant","content":"总价6元。"}]}
```

至少有两个独立 group；不要把同一会话的变体编造为不同来源。正文可以包含 JSON、代码和换行转义。

```bash
python lesson_34_sft_data_and_assistant_only_loss.py --data-jsonl my_sft.jsonl
python lesson_34_sft_data_and_assistant_only_loss.py --inspect --data-jsonl my_sft.jsonl
```

第一条仍使用教学 codec；第二条才使用真实配套分词器。所有拒绝均要检查原因，不要先删除校验只求“跑通”。

## 10. 独立练习：先不看后面的参考答案

检查器只给出成功或失败，不替你补代码。运行下面单元只是定义检查器，不代表你完成了练习。

In [23]:
def check_exercise_a(fn):
    tests = [([11,12,13,14],[0,0,1,1]), ([1,2,3,4,5],[0,1,0,1,1]), ([2,2,2],[0,0,1])]
    for ids,mask in tests:
        old = copy.deepcopy((ids,mask))
        require(fn(ids,mask)==labels_from_mask(ids,mask), "练习A标签不符")
        require((ids,mask)==old, "不能修改输入")
    for ids,mask in [([1,2],[0,0]), ([1,2],[1,1]), ([1,2],[0]), ([1,2],[0,2])]:
        try:
            fn(ids,mask)
        except ValueError:
            pass
        else:
            raise AssertionError("练习A未拒绝非法mask")

def check_exercise_b(fn):
    codec = ToyChatCodec()
    rows = [codec.encode(simple_record("A","B")), codec.encode(simple_record("longer","答案"))]
    old = copy.deepcopy(rows)
    got = fn(rows,codec.pad_id)
    require(got == collate_rows(rows,codec.pad_id), "练习B补齐、EOT或labels错误")
    require(rows == old, "练习B不能原地修改记录")
    for i,row in enumerate(rows):
        end = len(row["input_ids"])-1
        require(got["labels"][i][end]==codec.eot_id, "真实EOT被错误删除")
        require(got["attention_mask"][i][end]==1, "真实EOT的attention被错误关闭")

def check_exercise_c(fn):
    data = demo_records(); original = copy.deepcopy(data)
    train,valid = fn(data,{"order-R00","order-R03"})
    require({r["id"] for r in train+valid} == {r["id"] for r in data}, "有遗漏或重复")
    require({r["group_id"] for r in valid}=={"order-R00","order-R03"}, "验证group错误")
    audit_split(train,valid)
    require(data==original, "不能修改原数据")
    valid[0]["messages"][0]["content"] = "修改副本"
    require(data==original, "浅拷贝导致原数据被改写")

### 练习 A：根据 mask 创建同位 labels

实现 `make_labels_ex(ids, mask)`：目标位置复制原编号，其余填 `-100`；拒绝长度不等、非法 mask、首位目标和全忽略目标。不要改输入，也不要移位。

本题要自己写循环或列表推导，不要直接调用已写好的 `labels_from_mask`。

In [24]:
def make_labels_ex(ids, mask):
    raise NotImplementedError('由你实现：对齐标签、校验有效目标')

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(make_labels_ex)
    print('练习A通过')
else:
    print('练习A尚未验收')

练习A尚未验收


### 练习 B：在 PAD == EOT 时正确补齐

实现 `collate_ex(rows, pad_id)`，输出三个字段。保留真实 EOT 的监督；只忽略后补位置。不要复制普通 LM collator 的“重建全部 labels”逻辑。

In [25]:
def collate_ex(rows, pad_id):
    raise NotImplementedError('由你实现：基于原始长度的右补齐')

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(collate_ex)
    print('练习B通过')
else:
    print('练习B尚未验收')

练习B尚未验收


### 练习 C：来源级切分

实现 `split_groups_ex(records, valid_group_ids)`，返回 train/valid。每个来源只能在一侧；不得丢样本、复制样本或原地修改数据。验证 ID 集必须是已有来源的非空真子集。

允许调用 `audit_records` / `audit_split`，但请自己完成切分与深拷贝。

In [26]:
def split_groups_ex(records, valid_group_ids):
    raise NotImplementedError('由你实现：按来源group完整划分')

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(split_groups_ex)
    print('练习C通过')
else:
    print('练习C尚未验收')

练习C尚未验收


---
## 11. 参考答案

先完成上面的代码，再比较。参考函数使用独立名称，不会覆盖你的练习。特别检查：首位 labels、真实 EOT、原对象是否被修改。

In [27]:
def answer_a(ids, mask):
    require(len(ids) == len(mask) and len(ids) >= 2, "长度错误")
    require(all(type(x) is int and x >= 0 for x in ids), "非法token")
    require(all(type(x) in (int,bool) and x in (0,1) for x in mask), "非法mask")
    require(not mask[0] and any(mask[1:]), "无可用目标")
    return [ids[i] if mask[i] else IGNORE for i in range(len(ids))]

def answer_b(rows, pad_id):
    require(bool(rows), "空批次")
    for row in rows:
        validate_row(row)
    width = max(len(r["input_ids"]) for r in rows)
    result = {"input_ids":[], "attention_mask":[], "labels":[]}
    for row in rows:
        n = len(row["input_ids"])
        result["input_ids"].append(row["input_ids"][:] + [pad_id]*(width-n))
        result["attention_mask"].append([1]*n + [0]*(width-n))
        result["labels"].append(row["labels"][:] + [IGNORE]*(width-n))
    return result

def answer_c(records, valid_group_ids):
    audit_records(records)
    groups = {r["group_id"] for r in records}
    chosen = set(valid_group_ids)
    require(bool(chosen) and chosen < groups, "验证group必须是已有group的非空真子集")
    train, valid = [], []
    for record in records:
        (valid if record["group_id"] in chosen else train).append(copy.deepcopy(record))
    audit_split(train,valid)
    return train, valid

In [28]:
check_exercise_a(answer_a)
check_exercise_b(answer_b)
check_exercise_c(answer_c)
print('三个参考实现检查通过；不等于你的练习已完成。')

三个参考实现检查通过；不等于你的练习已完成。


## 12. 完整固定检查

下面是脚本同一组检查。标准库可验证数据逻辑；已安装 PyTorch 时增加 loss/梯度检查。未安装 PyTorch 的环境会明确跳过那一部分，不伪装成功。

这些检查不包含真实 Qwen tokenizer 的端到端执行，不代表 SFT 的任务效果。

In [29]:
class CheckBook:
    def __init__(self):
        self.names = []
    def ok(self, name, condition):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)
    def raises(self, name, fn, exception=ValueError):
        try:
            fn()
        except exception:
            self.names.append(name)
        else:
            raise AssertionError(name+"：没有按预期拒绝")

def run_checks(verbose=True):
    c = CheckBook(); codec = ToyChatCodec()
    rec = simple_record("A","B")
    validate_record(rec); c.ok("完整单轮通过",True)
    multi = demo_records()[0]
    validate_record(multi); c.ok("完整多轮通过",True)
    for name,mutator in [
        ("拒绝无id", lambda x:x.pop("id")),
        ("拒绝空group", lambda x:x.update(group_id="")),
        ("拒绝多模态", lambda x:x["messages"][0].update(content=[{"type":"text"}])),
        ("拒绝工具消息", lambda x:x["messages"][0].update(role="tool")),
        ("拒绝空答案", lambda x:x["messages"][-1].update(content=" \n")),
        ("拒绝未完成问答", lambda x:x["messages"].pop()),
        ("拒绝额外字段", lambda x:x.update(split="train")),
        ("拒绝assistant起始", lambda x:x["messages"][0].update(role="assistant")),
        ("拒绝孤立surrogate", lambda x:x["messages"][0].update(content="\ud800")),
    ]:
        bad = copy.deepcopy(rec); mutator(bad)
        c.raises(name,lambda b=bad:validate_record(b))
    spaces = simple_record("  x\n\ty ","```\n  a\n``` ")
    old = copy.deepcopy(spaces); codec.encode(spaces)
    c.ok("不strip代码与空白",spaces==old)
    data = demo_records(); c.ok("24个示例记录",len(data)==24)
    duplicate = copy.deepcopy(data[0]); duplicate["id"]="different"
    c.raises("拒绝重复对话",lambda:audit_records(data+[duplicate]))
    badid = copy.deepcopy(data[-1]); badid["id"]=data[0]["id"]
    c.raises("拒绝重复id",lambda:audit_records(data[:-1]+[badid]))
    state = random.getstate(); train,valid = split_by_group(data)
    c.ok("划分不消耗全局RNG",state==random.getstate())
    c.ok("按来源18/6划分",(len(train),len(valid))==(18,6))
    c.ok("输入顺序不影响划分",split_by_group(list(reversed(data)))==(train,valid))
    c.ok("两侧来源无交叉",not ({r['group_id'] for r in train}&{r['group_id'] for r in valid}))
    c.raises("逐行拆同来源会被发现",lambda:audit_split(data[::2],data[1::2]))
    other = copy.deepcopy(rec); other.update(id="other",group_id="other"); other["messages"][-1]["content"]="另一答案"
    c.raises("相同prompt跨split被发现",lambda:audit_split([rec],[other]))
    c.raises("单group无法切分",lambda:split_by_group([rec]))
    c.raises("非法比例被拒绝",lambda:split_by_group(data,valid_fraction=0))
    c.ok("全数据未因split改变",data==demo_records())
    row = codec.encode(rec)
    c.ok("可手算的序列",row["input_ids"]==[257,65,259,258,66,259])
    c.ok("只监督assistant内容和EOT",row["labels"]==[-100,-100,-100,-100,66,259])
    c.ok("prompt仍可见",row["attention_mask"]==[1]*6)
    c.ok("标签未提前移位",all(x==y for x,y in zip(row["input_ids"],row["labels"]) if y!=IGNORE))
    both,last = codec.encode(multi,"all"),codec.encode(multi,"last")
    c.ok("all与last输入完全相同",both["input_ids"]==last["input_ids"])
    c.ok("all覆盖两个答案",len(supervised_texts(both,codec))==2)
    c.ok("last仅覆盖末答案",len(supervised_texts(last,codec))==1)
    c.ok("last保留旧答案为上下文",len(last["input_ids"])==len(both["input_ids"]))
    c.ok("EOT作为学习目标",both["labels"][-1]==codec.eot_id)
    c.ok("header不作为学习目标",all(both["labels"][s["header_start"]]==IGNORE for s in both["spans"]))
    c.raises("拒绝未知监督策略",lambda:codec.encode(rec,"random"))
    c.raises("拒绝全忽略目标",lambda:labels_from_mask([1,2],[0,0]))
    c.raises("拒绝首位监督",lambda:labels_from_mask([1,2],[1,1]))
    c.raises("拒绝mask长度不匹配",lambda:labels_from_mask([1,2],[0]))
    c.raises("拒绝非二值mask",lambda:labels_from_mask([1,2],[0,2]))
    c.raises("拒绝布尔token ID",lambda:labels_from_mask([True,2],[0,1]))
    unicode_row = codec.encode(simple_record("你好🌍","世界"))
    c.ok("UTF8合并显示",supervised_texts(unicode_row,codec)==["世界<EOT>"])
    exact,_ = prepare_record(rec,codec,max_length=6)
    c.ok("刚好长度通过",exact is not None and len(exact["input_ids"])==6)
    dropped,log = prepare_record(rec,codec,max_length=5)
    c.ok("超长整条丢弃而非切断",dropped is None and log["status"]=="dropped")
    long = {"id":"long","group_id":"long","messages":[
        {"role":"system","content":"S"},{"role":"user","content":"X"*20},
        {"role":"assistant","content":"old answer"},{"role":"user","content":"U"},
        {"role":"assistant","content":"A"}]}
    oldlong = copy.deepcopy(long)
    shortened,event = prepare_record(long,codec,max_length=9,overflow="keep_latest_rounds")
    c.ok("完整轮次删除",event["removed_rounds"]==1 and len(shortened["input_ids"])==9)
    c.ok("system保留",shortened["effective_messages"][0]==long["messages"][0])
    c.ok("最终答案保留",supervised_texts(shortened,codec)==["A<EOT>"])
    c.ok("删历史风险显式记录",bool(event["warning"]))
    c.ok("长度处理不修改源数据",long==oldlong)
    too_small,_ = prepare_record(long,codec,max_length=8,overflow="keep_latest_rounds")
    c.ok("最后完整轮仍超长则丢弃",too_small is None)
    c.raises("拒绝自动token切断策略",lambda:prepare_record(rec,codec,overflow="truncate"))
    c.raises("拒绝bool长度",lambda:prepare_record(rec,codec,max_length=True))
    batch = collate_rows([row,codec.encode(simple_record("long","yes"))],codec.pad_id)
    c.ok("padding只增长度不移动答案",batch["labels"][0][:6]==row["labels"])
    c.ok("真实EOT不被当padding",batch["attention_mask"][0][5]==1 and batch["labels"][0][5]==259)
    c.ok("尾部padding忽略",all(x==IGNORE for x in batch["labels"][0][6:]))
    c.ok("padding attention为0",all(x==0 for x in batch["attention_mask"][0][6:]))
    c.ok("只向模型传三个字段",set(batch)=={"input_ids","labels","attention_mask"})
    c.ok("batch整倍数补齐",len(collate_rows([row],259,multiple=8)["input_ids"][0])==8)
    c.ok("目标分母不是输入长度",batch_statistics(batch)["supervised_targets"]==6)
    c.ok("按目标数聚合loss",merge_loss_summaries([(9,9),(9,1)])==1.8)
    c.raises("拒绝零分母摘要",lambda:merge_loss_summaries([(0,0)]))
    c.raises("拒绝空batch",lambda:collate_rows([],259))
    shifted = copy.deepcopy(row); shifted["labels"]=[-100,-100,-100,66,259,-100]
    c.raises("移位两次被发现",lambda:collate_rows([shifted],259))
    all_ignored = copy.deepcopy(row); all_ignored["labels"]=[-100]*6
    c.raises("全忽略样本被拒绝",lambda:collate_rows([all_ignored],259))
    rendered = boundary_fixture(multi["messages"])
    spans = qwen_char_spans(rendered,multi["messages"])
    c.ok("严格模板解析覆盖全部消息",len(spans)==5)
    c.ok("末assistant可能有模板空think",spans[-1]["body"].startswith("<think>\n\n</think>"))
    prior = boundary_fixture(multi["messages"][:3])
    c.ok("逐轮模板前缀并非总稳定",not rendered.startswith(prior))
    c.raises("多余生成前缀被拒绝",lambda:qwen_char_spans(rendered+"<|im_start|>assistant\n",multi["messages"]))
    c.raises("结束符缺失被拒绝",lambda:qwen_char_spans(rendered.replace("<|im_end|>","",1),multi["messages"]))
    c.raises("错误角色被拒绝",lambda:qwen_char_spans(rendered.replace("system","user",1),multi["messages"]))
    tinyspans=[{"role":"assistant","message_index":1,"char_start":2,"char_end":5}]
    mapped=offsets_to_token_spans([(0,2),(2,4),(4,5)],tinyspans,[7,8,9],9)
    c.ok("offset完整映射",mapped[0]["target_start"]==1 and mapped[0]["target_end"]==3)
    c.raises("跨边界token明确拒绝",lambda:offsets_to_token_spans([(0,3),(3,4),(4,5)],tinyspans,[7,8,9],9))
    c.raises("缺失结束token被拒绝",lambda:offsets_to_token_spans([(0,2),(2,4),(4,5)],tinyspans,[7,8,9],10))
    packing = packing_demo()
    c.ok("普通causal可跨文档读取",packing["second_document_reads_first_without_boundary_mask"])
    c.ok("隔离mask阻止跨文档读取",packing["with_isolation"] is False)
    prepared = prepare_dataset(data,codec)
    c.ok("管线保留24条",sum(v["records"] for v in prepared["summary"].values())==24)
    c.ok("默认不移除历史",all(e["removed_rounds"]==0 for e in prepared["events"]))
    c.ok("codec契约区分真实模型",prepared["codec_contract"]["kind"]=="teaching-byte-codec-not-Qwen")
    c.raises("长度处理后空集拒绝",lambda:prepare_dataset(data,codec,max_length=4))
    with tempfile.TemporaryDirectory() as directory:
        path=Path(directory)/"input.jsonl"
        write_jsonl(path,data)
        c.ok("JSONL往返不丢空白",read_jsonl(path)==data)
        path.write_text('{"id":"a","id":"b"}\n',encoding="utf-8")
        c.raises("JSON重复键拒绝",lambda:read_jsonl(path))
        path.write_text('\n',encoding="utf-8")
        c.raises("空JSONL行拒绝",lambda:read_jsonl(path))
        out,report=run_pipeline(directory,include_torch=False)
        c.ok("产物完整写入",all((out/name).exists() for name in ["train_tokens.jsonl","valid_tokens.jsonl","contract.json","manifest.json","report.json"]))
        c.ok("输出manifest标为教学codec",json.loads((out/"manifest.json").read_text())["scope"]=="teaching_codec")
        c.ok("输入顺序/样本完整保留",report["summary"]==prepared["summary"])
    for name,fn,checker in [("参考答案A",answer_a,check_exercise_a),("参考答案B",answer_b,check_exercise_b),("参考答案C",answer_c,check_exercise_c)]:
        checker(fn); c.ok(name,True)
    torch_result = torch_demo()
    if torch_result["status"]=="passed":
        c.ok("PyTorch loss有限",math.isfinite(torch_result["loss"]))
        c.ok("被忽略logit梯度为零",torch_result["ignored_logit_gradient_max"]==0)
        c.ok("prompt embedding仍有梯度",torch_result["prompt_embedding_gradient_norm"]>0)
        c.ok("改无监督logit不改loss",torch_result["loss"]==torch_result["loss_after_changing_ignored_logits"])
        import torch
        c.raises("移位后零目标拒绝",lambda:causal_token_loss(torch.zeros(1,3,5),torch.tensor([[1,-100,-100]])))
        logits=torch.randn(2,5,8,requires_grad=True)
        labels=torch.tensor([[-100,-100,3,2,-100],[-100,1,4,-100,-100]])
        loss,loss_sum,n=causal_token_loss(logits,labels)
        c.ok("有效分母为4",n==4)
        c.ok("sum与mean一致",torch.allclose(loss*4,loss_sum))
        loss.backward(); c.ok("loss保留梯度",logits.grad is not None and torch.isfinite(logits.grad).all())
    result={"passed":len(c.names),"names":c.names,"torch_demo":torch_result["status"],"real_qwen_tested":False}
    if verbose:
        print(f"固定检查通过：{len(c.names)} 项。真实Qwen分词器路径不属于这些检查。")
    return result

In [30]:
checks = run_checks()
print('PyTorch检查状态：', checks['torch_demo'])
print('真实Qwen纳入固定检查：', checks['real_qwen_tested'])

固定检查通过：95 项。真实Qwen分词器路径不属于这些检查。
PyTorch检查状态： passed
真实Qwen纳入固定检查： False


## 13. 验收与下一课

完成后应能说明：

- 哪些 token 进入 loss？是谁执行 next-token 移位？
- 为什么 user 仍在 attention 中，而其 labels 可以为 `-100`？
- PAD 和 EOT 相同时，为什么不能按编号屏蔽全部同值 token？
- 一个来源的多个派生样本为什么要保持在同一侧？
- 删除完整历史轮次为什么仍可能破坏语义？
- 为什么在真实路径中必须重新检查模板与 token 边界？

**本课没有训练模型，也没有提升成绩。** 它提供的是可以审计的数据契约。下一课进入第 35 课：将审查过的数据接入完整的 LoRA SFT 训练与留出评测，检查损失、生成、基座不变和保存重载，而不是只做两次梯度冒烟测试。

### 常用命令

```bash
python lesson_34_sft_data_and_assistant_only_loss.py --checks-only
python lesson_34_sft_data_and_assistant_only_loss.py
python lesson_34_sft_data_and_assistant_only_loss.py --policy last
python lesson_34_sft_data_and_assistant_only_loss.py --inspect
```

本课默认不打包多样本、不截断半个回答、不读模型权重。所有真实检查失败都会记录并抛出错误，不回退成教学结果冒充成功。

## 参考资料

1. [Transformers v4.57.3：聊天模板与特殊token](https://huggingface.co/docs/transformers/v4.57.3/en/chat_templating)。
2. [TRL v0.23.1：SFT目标、assistant-only及generation标记](https://huggingface.co/docs/trl/v0.23.1/sft_trainer)。
3. [TRL当前训练模板说明](https://huggingface.co/docs/trl/chat_templates)：有些模型会自动替换训练模板，具体行为取决于版本。
4. [固定Qwen3-0.6B模板配置](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/tokenizer_config.json)。
5. [固定Transformers内部CausalLM损失源码](https://raw.githubusercontent.com/huggingface/transformers/v4.57.3/src/transformers/loss/loss_utils.py)。
6. [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)。
7. [scikit-learn GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)。

模型来源和模板版本属于数据契约。将来换模型、换模板、换库版本或换监督策略，应重新验证，不直接复用之前的token文件。